# LangChain L6 — Level 5 — Conversation memory
Every OpsPilot so far forgets the previous turn. Users expect this to work:

```text
User : My name is Rahul and my customer id is C001.
Agent: Nice to meet you.
User : What plan am I on?          <- needs C001 from the previous turn
```

LangGraph **persistence** solves it. A *checkpointer* saves the agent state after every step,
keyed by a **thread id**. Invoking the same thread again loads the saved messages first.

```text
thread "rahul-1":   turn 1 -> checkpoint -> turn 2 -> checkpoint -> turn 3 ...
thread "priya-7":   turn 1 -> checkpoint ...                       (completely separate)
```

Two different memories are easy to confuse:

- **Short-term memory** = *this* conversation (the thread). This section.
- **Long-term memory** = what we know about the *user or application* across conversations. L7.

### Step 1 — Add a checkpointer and a thread id

`InMemorySaver` keeps checkpoints in RAM (fine for a notebook; production uses Postgres or
SQLite savers with the same interface). The thread id travels in `config["configurable"]`.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver     # LangGraph: saves state after every step

checkpointer = InMemorySaver()                              # LangGraph
opspilot_mem = create_agent(model=model, tools=READ_TOOLS, system_prompt=OPSPILOT_PROMPT, checkpointer=checkpointer)   # LangChain

rahul = {"configurable": {"thread_id": "rahul-1"}}          # LangGraph: the run config; thread_id selects the conversation

turn1 = opspilot_mem.invoke({"messages": [{"role": "user", "content": "My name is Rahul and my customer id is C001."}]}, rahul)
print("turn 1 :", text_of(turn1["messages"][-1])[:100])

turn2 = opspilot_mem.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, rahul)
print("turn 2 :", text_of(turn2["messages"][-1])[:100])
print("messages stored on this thread:", len(turn2["messages"]))

### Step 2 — Threads are isolated, and state is inspectable

A different thread id starts from nothing. `get_state()` reads the checkpoint without running
the agent, which is how a support dashboard would show "what does the agent currently know?".

In [ ]:
priya = {"configurable": {"thread_id": "priya-7"}}
other = opspilot_mem.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, priya)
print("new thread :", text_of(other["messages"][-1])[:100])

snapshot = opspilot_mem.get_state(rahul)                    # LangGraph: read the checkpoint without running
print("rahul's thread holds", len(snapshot.values["messages"]), "messages; last:", text_of(snapshot.values["messages"][-1])[:60])

### Step 3 — Long conversations: summarisation middleware

Threads grow. Eventually the history no longer fits the context window, or costs too much.
`SummarizationMiddleware` replaces older messages with a model-written summary once a trigger
is reached. This is our first **middleware**: behaviour inserted around the model call. L10
explains the mechanism fully.

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware   # LangChain: built-in middleware

opspilot_summ = create_agent(
    model=model, tools=READ_TOOLS, system_prompt=OPSPILOT_PROMPT,
    middleware=[SummarizationMiddleware(model=model, trigger=("messages", 6), keep=("messages", 2))],   # LangChain
    checkpointer=InMemorySaver(),                                                                          # LangGraph
)
long_thread = {"configurable": {"thread_id": "long-1"}}
for text in ["My name is Rahul.", "What is the weather in London?", "What is 12 * 12?", "And the weather in Mumbai?"]:
    out = opspilot_summ.invoke({"messages": [{"role": "user", "content": text}]}, long_thread)
    kinds = [m.type for m in out["messages"]]
    print(f"{text:32} -> {len(kinds):2} messages in state | summary present: {any('summary' in text_of(m).lower() for m in out['messages'] if m.type in ('system', 'human'))}")

### Recap

- **Problem seen:** each invocation started from an empty history.
- **Layer added:** a checkpointer plus a thread id (LangGraph persistence), and summarisation for long threads.
- **Evidence:** turn 2 answered from turn 1's facts; a different thread id knew nothing.